# 03 FedSTO Exact Reproduction

This notebook runs the paper-scale FedSTO reproduction path using the official SSFOD data setup repository and the EfficientTeacher YOLOv5L SSOD trainer stack. It is configured for the BDD100K non-IID weather setup used in the paper: cloudy server labels and overcast/rainy/snowy unlabeled clients.

In [ ]:
from pathlib import Path
import json
import subprocess
import sys
from typing import Optional


def find_project_root(start: Optional[Path] = None) -> Path:
    start = Path.cwd().resolve() if start is None else Path(start).resolve()
    markers = (
        "setup_fedsto_exact_reproduction.py",
        "run_fedsto_efficientteacher_exact.py",
        "03_fedsto_exact_reproduction.ipynb",
    )
    candidate_dirs = []
    for base in (start, *start.parents):
        candidate_dirs.extend([
            base,
            base / "navigating_data_heterogeneity",
            base / "Object_Detection" / "navigating_data_heterogeneity",
            base / "masters_research" / "navigating_data_heterogeneity",
        ])
    for candidate in candidate_dirs:
        if all((candidate / marker).exists() for marker in markers[:2]) or any((candidate / marker).exists() for marker in markers):
            return candidate.resolve()
    for pattern in (
        "*/setup_fedsto_exact_reproduction.py",
        "*/*/setup_fedsto_exact_reproduction.py",
    ):
        matches = list(start.glob(pattern))
        if matches:
            return matches[0].parent.resolve()
    return start


PROJECT_ROOT = find_project_root()
SETUP_SCRIPT = PROJECT_ROOT / "setup_fedsto_exact_reproduction.py"
RUN_SCRIPT = PROJECT_ROOT / "run_fedsto_efficientteacher_exact.py"
WORK_ROOT = PROJECT_ROOT / "efficientteacher_fedsto"

missing = [path for path in (SETUP_SCRIPT, RUN_SCRIPT) if not path.exists()]
if missing:
    raise FileNotFoundError("Missing required scripts:\n" + "\n".join(str(path) for path in missing))

print("PROJECT_ROOT:", PROJECT_ROOT)
print("SETUP_SCRIPT:", SETUP_SCRIPT)
print("RUN_SCRIPT:", RUN_SCRIPT)

## 1. Repair/Rebuild Paper-Scale Data

Run this if EfficientTeacher reports many corrupt images or no labels. It rebuilds `data_paper20k` from the local BDD100K raw dataset and removes stale EfficientTeacher cache files. Use `COPY_IMAGES=True` for Docker or moved workspaces.


In [ ]:
COPY_IMAGES = True

cmd = [sys.executable, str(PROJECT_ROOT / "prepare_bdd100k_paper20k.py")]
if COPY_IMAGES:
    cmd.append("--copy-images")
subprocess.run(cmd, cwd=PROJECT_ROOT, check=True)

for cache_path in (PROJECT_ROOT / "efficientteacher_fedsto" / "data_lists").glob("*.cache"):
    cache_path.unlink()
    print("removed stale cache:", cache_path)


## 2. Build Paper-Scale Data Interface

This creates EfficientTeacher-compatible list files and YAML configs from the BDD100K paper-scale split already prepared in `data_paper20k`.

In [ ]:
subprocess.run([sys.executable, str(SETUP_SCRIPT)], cwd=PROJECT_ROOT, check=True)
manifest = json.loads((WORK_ROOT / "manifest.json").read_text())
manifest

## 3. Confirm Paper Conditions

In [ ]:
summary = {
    "server_train_images": manifest["server"]["train_images"],
    "server_val_images": manifest["server"]["val_images"],
    "client_images": {f"client_{c['id']}_{c['weather']}": c["images"] for c in manifest["clients"]},
    "classes": manifest["classes"],
    "schedule": manifest["paper_schedule"],
    "official_ssfod_sha": manifest["official_ssfod_sha"],
    "efficientteacher_sha": manifest["efficientteacher_sha"],
}
summary

## 4. EfficientTeacher Runtime Dependency Check

Run this before starting real training. Dry-run only checks command/config generation; this cell checks imports used by EfficientTeacher.


In [ ]:
import importlib.util

required = {
    "yaml": "PyYAML",
    "cv2": "opencv-python",
    "thop": "thop",
    "tensorboard": "tensorboard",
    "sklearn": "scikit-learn",
}
missing = [package for module, package in required.items() if importlib.util.find_spec(module) is None]

pkg_resources_shim = PROJECT_ROOT / "vendor" / "efficientteacher" / "pkg_resources.py"
if importlib.util.find_spec("pkg_resources") is None and not pkg_resources_shim.exists():
    missing.append("setuptools<81")

if missing:
    print("Missing packages:", missing)
    print("Install in this kernel/environment, then rerun this cell:")
    print(f"{sys.executable} -m pip install " + " ".join(missing))
    raise ModuleNotFoundError("Missing EfficientTeacher dependencies: " + ", ".join(missing))
else:
    print("EfficientTeacher runtime dependency check OK")
    if pkg_resources_shim.exists():
        print("Using vendor pkg_resources shim:", pkg_resources_shim)


## 5. Dry-Run Command Check

This verifies the runner and runtime config generation without starting training.

In [ ]:
subprocess.run([
    sys.executable, str(RUN_SCRIPT),
    "--dry-run",
    "--warmup-epochs", "1",
    "--phase1-rounds", "1",
    "--phase2-rounds", "1",
], cwd=PROJECT_ROOT, check=True)

## 6. Full Paper Reproduction Run

This is the production run: 50 supervised warm-up epochs, 100 selective-training rounds, and 150 full-parameter orthogonal rounds. It downloads the official EfficientTeacher YOLOv5L checkpoint if it is not already present.

In [ ]:
RUN_FULL_REPRODUCTION = True
BATCH_SIZE = "64"  # 2 GPUs x 32 images/GPU.
DATALOADER_WORKERS = "0"  # Keep 0 when Docker /dev/shm is small.
NUM_GPUS = "2"
MASTER_PORT = "29500"

# Full FedSTO reproduction: 50 warmup epochs + 100 phase-1 rounds + 150 phase-2 rounds.
# 2-GPU DDP is much faster than single-GPU DataParallel for this EfficientTeacher codebase.

cmd = [
    sys.executable, str(RUN_SCRIPT),
    "--warmup-epochs", "50",
    "--phase1-rounds", "100",
    "--phase2-rounds", "150",
    "--batch-size", BATCH_SIZE,
    "--workers", DATALOADER_WORKERS,
    "--gpus", NUM_GPUS,
    "--master-port", MASTER_PORT,
]

if RUN_FULL_REPRODUCTION:
    log_path = PROJECT_ROOT / "efficientteacher_fedsto" / "full_reproduction_latest.log"
    log_path.parent.mkdir(parents=True, exist_ok=True)
    print("Running:", " ".join(cmd))
    print("Log:", log_path)
    with log_path.open("w", encoding="utf-8") as log_file:
        process = subprocess.Popen(
            cmd,
            cwd=PROJECT_ROOT,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
        )
        assert process.stdout is not None
        for line in process.stdout:
            print(line, end="")
            log_file.write(line)
        return_code = process.wait()
    if return_code != 0:
        raise subprocess.CalledProcessError(return_code, cmd)
else:
    print("Set RUN_FULL_REPRODUCTION = True to start the full FedSTO reproduction run.")


## 7. Inspect Outputs

In [ ]:
history_path = WORK_ROOT / "history.json"
if history_path.exists():
    history = json.loads(history_path.read_text())
    print("completed rounds:", len(history))
    print("latest:", history[-1])
else:
    print("history.json is not present yet; full training has not completed a round.")

print("runs:", WORK_ROOT / "runs")
print("global checkpoints:", WORK_ROOT / "global_checkpoints")
print("client EMA states:", WORK_ROOT / "client_states")